In [5]:
from inference_funcs import load_bit_phoneme_model, evaluate_model, decode_outputs
from dataset import getDatasetLoaders
import numpy as np
from edit_distance import SequenceMatcher
from typing import Any, Iterable, Sequence
import re 
from g2p_en import G2p
import numpy as np

In [6]:
def postprocess_topk(topk_labels_indices_to_keep: np.ndarray, blank_token: str = "~") -> np.ndarray:
    """
    For each row:
    - If the blank token is present, remove it.
    - Otherwise, remove the last element.
    
    Always returns an array with one fewer column than input.
    """
    processed_rows = []
    for row in topk_labels_indices_to_keep:
        if blank_token in row:
            # remove first occurrence of blank
            new_row = [tok for tok in row if tok != blank_token]
        else:
            new_row = row[:-1]
        processed_rows.append(new_row)
    return np.array(processed_rows, dtype=object)

def topk_labels(logits: np.ndarray, K: int, vocab: list, apply_ctc_rule: bool) -> np.ndarray:
    
    """
    Args:
        logits: np.ndarray of shape (T, N) where
                N = 1 + len(vocab) (index 0 = CTC blank, rest follow vocab order)
        K: number of top tokens to return
        vocab: list of labels (phonemes or characters)
        apply_ctc_rule: removes blanks and repeats not separated by a blank

    Returns:
        np.ndarray of shape (T, K) with label strings
    """
    
    # Build id -> label mapping
    
    blank_token = "~"
    
    id2label = [blank_token] + vocab

    # Get top-K indices per timestep
    topk_ids = np.argsort(logits, axis=1)[:, -(K+1):][:, ::-1]
    topk_logits = np.sort(logits, axis=1)[:, -(K+1):][:, ::-1]

    # Map to labels
    topk_labels = np.vectorize(lambda i: id2label[i])(topk_ids)
    
    if apply_ctc_rule:
        
        top1_ids = np.argsort(logits, axis=1)[:, -1:][:, ::-1]
        
        blank_indices = np.argwhere(top1_ids == 0).squeeze()
        
        # returns indices where the next index is the same
        repeating_indices = np.argwhere(top1_ids[:-1] == top1_ids[1:]).squeeze()
        
        indices_to_remove = np.union1d(blank_indices, repeating_indices+1).squeeze()
        
        indices_to_keep = np.setdiff1d(np.arange(len(top1_ids)), indices_to_remove)
        
        topk_labels_indices_to_keep = postprocess_topk(topk_labels[indices_to_keep])
        
        # remove the blank token for rows which have it, for rows which don't have it remove the last element for topk_labels_indices_to_keep
        
        return topk_labels_indices_to_keep, indices_to_remove.shape[0], indices_to_keep.shape[0], indices_to_keep
    
    else:
        
        return topk_labels, topk_logits

def rows_to_text_dual(
    arr1: Any,
    arr2: Any,
    elem_sep: str = " ",
    row_sep: str = "\n",
    side_sep: str = " | "
) -> str:
    """
    Convert two 2-D arrays (or list-of-lists) to a single string with side-by-side rows.
    - Each element in a row is joined with `elem_sep`.
    - Each pair of rows is joined with `side_sep`.
    - Each line is joined with `row_sep` (newline by default).
    """
    # Support numpy arrays or list-of-lists
    rows1 = arr1.tolist() if isinstance(arr1, np.ndarray) else arr1
    rows2 = arr2.tolist() if isinstance(arr2, np.ndarray) else arr2
    
    if len(rows1) != len(rows2):
        raise ValueError("Both arrays must have the same number of rows")
    
    lines = []
    for r1, r2 in zip(rows1, rows2):
        if isinstance(r1, np.ndarray): r1 = r1.tolist()
        if isinstance(r2, np.ndarray): r2 = r2.tolist()
        left = elem_sep.join(map(str, r1))
        right = elem_sep.join(map(str, r2))
        lines.append(left + side_sep + right)
    
    return row_sep.join(lines)


def rows_to_text(arr: Any, elem_sep: str = " ", row_sep: str = "\n") -> str:
    """
    Convert a 2-D numpy array (or list of lists) to a single string.
    - Each element in a row is joined with `elem_sep`.
    - Each row is joined with `row_sep` (newline by default).
    """
    # Support ndarray or list-of-lists
    rows: Iterable = arr.tolist() if isinstance(arr, np.ndarray) else arr
    
    lines = []
    for r in rows:
        if isinstance(r, np.ndarray):
            r = r.tolist()
        lines.append(elem_sep.join(map(str, r)))
    
    return row_sep.join(lines)

def grapheme_to_phoneme(sentence):
    
    phonemes = ""
    
    for p in g2p(sentence):
        
        if p==' ':
            phonemes += p
        p = re.sub(r'[0-9]', '', p)  # Remove stress
        
        if re.match(r'[A-Z]+', p):  # Only keep phonemes
            phonemes += p
    
    return phonemes


In [7]:
# Phone definitions and mappings
PHONE_DEF = [
    'AA', 'AE', 'AH', 'AO', 'AW',
    'AY', 'B',  'CH', 'D', 'DH',
    'EH', 'ER', 'EY', 'F', 'G',
    'HH', 'IH', 'IY', 'JH', 'K',
    'L', 'M', 'N', 'NG', 'OW',
    'OY', 'P', 'R', 'S', 'SH',
    'T', 'TH', 'UH', 'UW', 'V',
    'W', 'Y', 'Z', 'ZH'
]

PHONE_DEF_SIL = PHONE_DEF + ["<>"]

CHAR_VOCAB = [
    "<sp>",          # space token
    "!", ",", ".", "?", "'",   # punctuation (incl. apostrophe)
] + [chr(i) for i in range(ord('a'), ord('z') + 1)]  # 'a'..'z'


In [24]:
datasets = ['ptDecoder_ctc_both_char_phoneme']
models = ["time_masked_transfomer_characters_phonemes_80ms_seed_0"]

dataset_base_path = '/data/neural_data/'
model_base_path = '/data/models/'

partition = 'test'

device = 'cuda'

phoneme_outputs_train = []
char_outputs_train = []

phoneme_outputs_train_logits = []
char_outputs_train_logits = []

phoneme_frames_removed_train = 0
phoneme_frames_kept_train = 0
phoneme_indices_kept_train = []

ground_truth_sentences_arr_train = []
decoded_sentences_arr_train = []



for fn, (dataset, model) in enumerate(zip(datasets, models)):

    data_file = f"{dataset_base_path}{dataset}"

    trainLoaders, testLoaders, loadedData = getDatasetLoaders(
            data_file, 8, None, 
            False
    )
    
    bit_phoneme_filepath = f"{model_base_path}{model}"
    model, args = load_bit_phoneme_model(bit_phoneme_filepath)
    
    
    model = model.to(device)
    
    outputs, cer, per_day_cer, cer, per_day_cer2 = evaluate_model(model, loadedData, args, partition=partition, device='cuda')
    
    decoded_strs_train = decode_outputs(outputs['decodedSeqs2'], mode='phoneme')
    decoded_strs_char = decode_outputs(outputs['decodedSeqs'], mode='char')
    
    true_seq_strs_train = outputs['transcriptions']
        
    for phoneme_output in outputs['logits2']:
        
        topk_phones, topk_phones_logits = topk_labels(phoneme_output, K=10, vocab=PHONE_DEF_SIL, apply_ctc_rule=False)
        interleaved_phones = np.empty((topk_phones.shape[0], topk_phones.shape[1]*2), dtype=topk_phones.dtype)
        interleaved_phones[:, 0::2] = topk_phones
        interleaved_phones[:, 1::2] = topk_phones_logits
        phoneme_outputs_train.append(interleaved_phones)

    for char_output in outputs['logits']:
        
        topk_chars, topk_chars_logits = topk_labels(char_output, K=10, vocab=CHAR_VOCAB, apply_ctc_rule=False)
        interleaved_char = np.empty((topk_chars.shape[0], topk_chars.shape[1]*2), dtype=topk_chars.dtype)
        interleaved_char[:, 0::2] = topk_chars
        interleaved_char[:, 1::2] = topk_chars_logits
        char_outputs_train.append(interleaved_char)

CER DAY 0: 0.373233
CER2 DAY 0: 0.354586
CER DAY 1: 0.288823
CER2 DAY 1: 0.275862
CER DAY 2: 0.255659
CER2 DAY 2: 0.238019
CER DAY 3: 0.275168
CER2 DAY 3: 0.254317
CER DAY 4: 0.152788
CER2 DAY 4: 0.124664
CER DAY 5: 0.110843
CER2 DAY 5: 0.099812
CER DAY 6: 0.135662
CER2 DAY 6: 0.116367
CER DAY 7: 0.171516
CER2 DAY 7: 0.152406
CER DAY 8: 0.170501
CER2 DAY 8: 0.142629
CER DAY 9: 0.179943
CER2 DAY 9: 0.178333
CER DAY 10: 0.161169
CER2 DAY 10: 0.143482
CER DAY 11: 0.190287
CER2 DAY 11: 0.191176
CER DAY 12: 0.160377
CER2 DAY 12: 0.139405
CER DAY 13: 0.144928
CER2 DAY 13: 0.134768
CER DAY 14: 0.109924
CER2 DAY 14: 0.100803
CER DAY 15: 0.119756
CER2 DAY 15: 0.105596
CER DAY 16: 0.139493
CER2 DAY 16: 0.114286
CER DAY 17: 0.168908
CER2 DAY 17: 0.143139
CER DAY 18: 0.125465
CER2 DAY 18: 0.108647
CER DAY 19: 0.126476
CER2 DAY 19: 0.109576
CER DAY 20: 0.106255
CER2 DAY 20: 0.100503
CER DAY 21: 0.143902
CER2 DAY 21: 0.127460
CER DAY 22: 0.130709
CER2 DAY 22: 0.126969
CER DAY 23: 0.167993
CER2 DAY 2

In [35]:
np.set_printoptions(precision=4, suppress=True)
print(phoneme_outputs_train[400][5:20, 0:10])

[['~' '20' '<>' '2.' 'AH' '-0' 'S' '-1' 'IY' '-2']
 ['~' '21' '<>' '3.' 'AH' '2.' 'V' '-0' 'S' '-0']
 ['~' '18' 'V' '9.' '<>' '4.' 'F' '2.' 'AH' '1.']
 ['V' '21' 'F' '11' '~' '7.' 'P' '4.' 'W' '4.']
 ['~' '11' 'V' '9.' 'R' '7.' 'OY' '4.' 'EH' '3.']
 ['OY' '15' 'EH' '13' 'AY' '11' 'AH' '11' 'AO' '11']
 ['~' '12' 'R' '9.' 'D' '7.' 'AH' '2.' 'OY' '0.']
 ['D' '13' '~' '10' 'R' '8.' 'AH' '4.' 'T' '3.']
 ['D' '15' '~' '8.' 'AH' '7.' 'T' '7.' 'IY' '4.']
 ['D' '13' 'IY' '12' '~' '8.' 'AH' '8.' 'T' '7.']
 ['IY' '16' 'D' '9.' 'EY' '7.' '~' '7.' 'AH' '6.']
 ['IY' '12' '<>' '10' '~' '9.' 'EY' '7.' 'D' '5.']
 ['<>' '19' 'D' '8.' '~' '7.' 'EY' '3.' 'IY' '3.']
 ['<>' '16' '~' '11' 'D' '8.' 'F' '7.' 'R' '0.']
 ['~' '11' 'AE' '11' 'EY' '7.' 'F' '6.' 'EH' '5.']]


In [36]:
Yeah 21/2.5

8.4

In [42]:
import torch
torch.set_printoptions(precision=6, sci_mode=False)
torch.nn.functional.log_softmax(torch.tensor([20.0, 10.0]))

/tmp/ipykernel_897670/3633613450.py:3: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  torch.nn.functional.log_softmax(torch.tensor([20.0, 10.0]))


tensor([    -0.000045,    -10.000046])

In [30]:
char_outputs_train[400]

array([['~', '<sp>', 't', 'e', 'a', 'w', 'r', 'd', 'b', 'y', 's'],
       ['~', '<sp>', 't', 'e', 'a', 'w', 'r', 'd', 'y', 'b', 'h'],
       ['~', '<sp>', 't', 'e', 'a', 'w', 'y', 'd', 'r', 'b', 's'],
       ['~', '<sp>', 't', 'e', 'a', 'd', 'w', 'r', 'i', 'y', 's'],
       ['~', '<sp>', 't', 'e', 'a', 'w', 'd', 'i', 'r', 's', 'y'],
       ['~', '<sp>', 'a', 'e', 't', 's', 'i', 'w', 'd', 'r', 'f'],
       ['~', '<sp>', 'a', 'e', 't', 'v', 's', 'i', 'f', 'd', 'w'],
       ['~', 'v', '<sp>', 'f', 'a', 't', 'e', 'w', 'p', 'i', 'r'],
       ['v', 'f', '~', 'p', 'w', 'r', 'b', 'a', 'o', 'e', 'h'],
       ['~', 'o', 'v', 'r', 'a', 'i', 'e', 'h', 'l', 'f', 'u'],
       ['o', 'i', 'a', 'e', 'u', 'r', '~', 'd', 'y', 'v', 'h'],
       ['~', 'r', 'd', 'i', 'u', 'o', 'a', 't', 'e', 'l', 'n'],
       ['d', '~', 'r', 'i', 'o', 't', 'u', 'a', 'n', 'e', 'g'],
       ['d', '~', 'i', 't', 'o', 'e', 'r', 'a', 'n', 'u', 'h'],
       ['d', '~', 't', 'i', 'a', 'e', 'y', 'o', '<sp>', 'u', 'n'],
       ['y', 

In [40]:
def load_lexicon_simple(filepath):
    """
    Loads a lexicon file into a dictionary using a dictionary comprehension.

    Args:
        filepath (str): The path to the lexicon text file.

    Returns:
        dict: A dictionary with words as keys and lists of integers as values.
    """
    try:
        with open(filepath, 'r') as f:
            return {
                parts[0]: [int(num) for num in parts[1:]]
                for line in f if (parts := line.strip().split())
            }
    except FileNotFoundError:
        print(f"Error: The file at '{filepath}' was not found.")
        return None

# --- Usage Example ---
file_path = '/data/lm/lexicon_numbers.txt'
word_to_numbers = load_lexicon_simple(file_path)
friday_phones = word_to_numbers['FRIDAY']


def idToPhoneme(i: int) -> str:
    return PHONE_DEF_SIL[i]

In [41]:
for ph in friday_phones:
    print(idToPhoneme(ph-1))

F
R
AY
D
IY


In [13]:
print(decoded_strs_train[400], true_seq_strs_train[400])
print(decoded_strs_char[400], true_seq_strs_train[400])

VOYDIY AEFTERTNUWNT AET FAYV THLEYDIY  friday afternoon at five thirty
vody afterint at five lody  friday afternoon at five thirty


In [ ]:
topk_labels(phoneme_output, K=10, vocab=PHONE_DEF_SIL, apply_ctc_rule=True)

In [119]:
sorted_idxs_train = np.argsort(true_seq_strs_train)
sorted_idxs_cross_val = np.argsort(ground_truth_sentences_arr)

true_seq_strs_train_sorted = [true_seq_strs_train[i] for i in sorted_idxs_train]
ground_truth_sentences_arr_sorted = [ground_truth_sentences_arr[i] for i in sorted_idxs_cross_val]

decoded_strs_train_stored = [decoded_strs_train[i] for i in sorted_idxs_train]
decoded_sentences_arr_sorted = [decoded_sentences_arr[i] for i in sorted_idxs_cross_val]

phoneme_outputs_sorted = [phoneme_outputs[i] for i in sorted_idxs_cross_val]
phoneme_outputs_train_sorted = [phoneme_outputs_train[i] for i in sorted_idxs_train]

import random 
random.seed(42)
shuffled_indices = list(range(len(phoneme_outputs_sorted)))
random.shuffle(shuffled_indices)

# these two lists will be the same
true_seq_strs_train_shuffled = [true_seq_strs_train_sorted[i] for i in shuffled_indices]
ground_truth_sentences_arr_shuffled = [ground_truth_sentences_arr_sorted[i] for i in shuffled_indices]

decoded_strs_train_shuffled = [decoded_strs_train_stored[i] for i in shuffled_indices]
decoded_sentences_arr_shuffled = [decoded_sentences_arr_sorted[i] for i in shuffled_indices]

phoneme_outputs_shuffled = [phoneme_outputs_sorted[i] for i in shuffled_indices]
phoneme_outputs_train_shuffled = [phoneme_outputs_train_sorted[i] for i in shuffled_indices]

# alternate between high accuracy and lower accuracy sentences
block_size = 800
combined_ground_truth = []
combined_phoneme_outputs = []
for i in range(0, 8800, block_size):
    
    combined_ground_truth.extend(true_seq_strs_train_shuffled[i:i+block_size])
    combined_ground_truth.extend(ground_truth_sentences_arr_shuffled[i:i+block_size])
    
    combined_phoneme_outputs.extend(phoneme_outputs_train_shuffled[i:i+block_size])
    combined_phoneme_outputs.extend(phoneme_outputs_shuffled[i:i+block_size])
    
print(len(combined_phoneme_outputs))
print(len(combined_ground_truth))


17600
17600


In [120]:
import json
with open(f'/data2/jsonl/combined_gt.json', "w") as f:
    json.dump(combined_ground_truth, f)

In [121]:
compute_per_sentence_cer = False
cer_per_sentence = []
if compute_per_sentence_cer:
    for true_seq, decoded in zip(outputs['trueSeqs2'], outputs['decodedSeqs2']):
        
        ed = SequenceMatcher(a=true_seq.tolist(), b=decoded.tolist()).distance()
        cer_per_sentence.append(ed/len(true_seq))
        
    np.save('/data2/perf_metrics/cer_per_sentence', cer_per_sentence)

In [123]:
import json
from pathlib import Path

OUT_JSONL = '/data2/jsonl/cross_val.jsonl'
eval_mode = False

USER_HDR = "<|start_header_id|>user<|end_header_id|>"
ASST_HDR = "<|start_header_id|>assistant<|end_header_id|>"
EOT      = "<|eot_id|>"   # include if your format expects it

phoneme_array = phoneme_outputs_shuffled
ground_truth_array = ground_truth_sentences_arr_shuffled

prompt = (
    "You are helping decode speech from neural activity to restore communication for a paralyzed patient. "
    "For each non-overlapping neural time bin, a neural network provides the 10 most probable tokens. "
    "Tokens consist of ARPAbet phonemes and the space character, written as <>. "
    "Each line contains the 10 candidate tokens for one time bin, ordered from most to least likely. "
    "The output contains many mistakes and has no knowledge of English spelling or grammar. "
    "Your task is to reconstruct the intended English sentence by correcting these mistakes. "
    "Consider all candidate tokens, not just the most likely ones, when inferring the intended words. "
    "Convert ARPAbet phonemes into English words. "
    "Produce a coherent, grammatical sentence. "
    "Output only the intended sentence, with no additional explanations or metadata."
)

with Path(OUT_JSONL).open("w", encoding="utf-8") as fout:
    
    for idx in range(len(phoneme_array)):
        
        topk_phones = rows_to_text(phoneme_array[idx])
        
        if eval_mode:
            ground_truth = ""
            
        else:
            gt_sentence = ground_truth_array[idx]
            ground_truth = f"{gt_sentence}{EOT}"
        
        msg = (
            f"{USER_HDR}\n\n"
            f"{prompt}\n\n"
            f"{topk_phones}\n\n"
            f"{ASST_HDR}\n\n"
            f"{ground_truth}"
        )
        
        obj = {"text": msg}
        
        fout.write(json.dumps(obj, ensure_ascii=False) + "\n")

In [124]:
print(msg)

<|start_header_id|>user<|end_header_id|>

You are helping decode speech from neural activity to restore communication for a paralyzed patient. For each non-overlapping neural time bin, a neural network provides the 10 most probable tokens. Tokens consist of ARPAbet phonemes and the space character, written as <>. Each line contains the 10 candidate tokens for one time bin, ordered from most to least likely. The output contains many mistakes and has no knowledge of English spelling or grammar. Your task is to reconstruct the intended English sentence by correcting these mistakes. Consider all candidate tokens, not just the most likely ones, when inferring the intended words. Convert ARPAbet phonemes into English words. Produce a coherent, grammatical sentence. Output only the intended sentence, with no additional explanations or metadata.

HH Y DH JH SH L G S K CH
IY IH AE EH UW EY AH R ER Z
Z S D R T IY V JH SH <>
<> Z S D T IY IH NG JH R
S T <> D HH R G K Z F
R T S W D EH HH L IY G
EH